# Role-Based Access Control (RBAC) with Apache Polaris

This notebook demonstrates fine-grained **Role-Based Access Control (RBAC)** in Apache Polaris
using the existing medallion architecture.

**Prerequisites:** Run `01_setup.ipynb` first to ensure the catalog, namespaces, and tables exist.

---

## Scenario: The Partial Gold Share

Two product teams share the same Polaris catalog (`lakehouse`):

| Team | Role | Principal | Access |
|------|------|-----------|--------|
| Product B — E-Commerce Orders | Data Engineer | `tariq` | Full read + write on `bronze`, `silver`, `gold` |
| Product A — Inventory & Logistics | Data Scientist | `ahmed` | Read-only on entire `gold` namespace |
| Product A — Inventory & Logistics | Data Scientist | `layla` | Read-only on `gold.daily_sales_summary` table only |

**Product B** owns all three medallion tables:
- `bronze.raw_orders` — raw ingestion events
- `silver.cleansed_orders` — cleaned and deduplicated orders
- `gold.daily_sales_summary` — business aggregations

This is enforced entirely at the **Polaris catalog layer** — no application-level guards needed.

---
## Step 1 — Authentication & Setup

In [ ]:
import requests

# Polaris
POLARIS_URL = "http://polaris:8181"
POLARIS_CLIENT_ID = "root"
POLARIS_CLIENT_SECRET = "polaris-secret"
CATALOG_NAME = "lakehouse"

# SeaweedFS S3
S3_ENDPOINT = "http://seaweedfs:8333"
S3_ACCESS_KEY = "lakehouse-admin"
S3_SECRET_KEY = "lakehouse-secret-key"
S3_REGION = "us-east-1"


def get_polaris_token(client_id=POLARIS_CLIENT_ID, client_secret=POLARIS_CLIENT_SECRET):
    resp = requests.post(
        f"{POLARIS_URL}/api/catalog/v1/oauth/tokens",
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": "PRINCIPAL_ROLE:ALL",
        },
    )
    if resp.ok:
        return resp.json()["access_token"]
    raise Exception(f"Failed to get token: {resp.text}")


ROOT_TOKEN = get_polaris_token()
root_headers = {"Authorization": f"Bearer {ROOT_TOKEN}", "Content-Type": "application/json"}
print("Authenticated to Polaris as root")

### Step 1b — Enable Credential Vending on the Polaris Catalog

The `lakehouse` catalog was originally registered with `stsUnavailable: true`, which prevents
Polaris from vending temporary S3 credentials. SeaweedFS does support STS at the same S3
endpoint, so we patch the catalog to re-enable credential vending.

Polaris credential vending works as follows:
1. PyIceberg sends `X-Iceberg-Access-Delegation: vended-credentials` when calling `loadTable`.
2. Polaris calls the configured STS endpoint (`AssumeRole`) to generate a short-lived token.
3. Polaris returns S3 credentials scoped to that table's location inside the `LoadTableResponse`.
4. PyIceberg uses those credentials for all Parquet file access — no static keys needed.

In [ ]:
# Get the current catalog version (required for optimistic concurrency on PATCH)
catalog_resp = requests.get(
    f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}",
    headers=root_headers,
)
catalog_resp.raise_for_status()
catalog_data = catalog_resp.json()
entity_version = catalog_data["entityVersion"]
current_storage = catalog_data["storageConfigInfo"]

print(f"Current catalog entity version : {entity_version}")
print(f"Current stsUnavailable         : {current_storage.get('stsUnavailable', False)}")

# Patch: set stsUnavailable to False so Polaris can call the SeaweedFS STS endpoint
patch_payload = {
    "currentEntityVersion": entity_version,
    "storageConfigInfo": {
        "storageType": "S3",
        "stsUnavailable": False,
        "pathStyleAccess": True,
        "endpoint": S3_ENDPOINT,
        "region": S3_REGION,
        "allowedLocations": current_storage["allowedLocations"],
    },
}

patch_resp = requests.put(
    f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}",
    headers=root_headers,
    json=patch_payload,
)

if patch_resp.ok:
    print("Catalog patched: stsUnavailable -> False (credential vending enabled)")
else:
    print(f"PATCH failed ({patch_resp.status_code}): {patch_resp.text}")

---
## Step 2 — Polaris RBAC Concepts

Polaris enforces access through a layered role hierarchy:

```
Principal (layla)
  └── Principal Role (inventory_data_scientist)
        └── Catalog Role (gold_sales_table_reader)
              └── TABLE privilege on gold.daily_sales_summary only
```

Privileges can be granted at three granularities:

| Scope | Example | Effect |
|-------|---------|--------|
| Catalog | `CATALOG_MANAGE_CONTENT` | Entire catalog |
| Namespace | `TABLE_READ_DATA` on `gold` | All current and future tables in that namespace |
| Table | `TABLE_READ_DATA` on `gold.daily_sales_summary` | That one table only |

---
## Step 3 — Create Principals

In [ ]:
def create_principal(name):
    """Create a Polaris principal, recreating it if it already exists."""
    r = requests.post(
        f"{POLARIS_URL}/api/management/v1/principals",
        headers=root_headers,
        json={"principal": {"name": name}},
    )
    if r.status_code in (200, 201):
        creds = r.json()["credentials"]
        print(f"Created principal '{name}'  (clientId: {creds['clientId']})")
        return creds
    if r.status_code == 409:
        print(f"Principal '{name}' already exists — recreating to obtain fresh credentials...")
        requests.delete(f"{POLARIS_URL}/api/management/v1/principals/{name}", headers=root_headers)
        return create_principal(name)
    raise Exception(f"Failed to create principal '{name}': {r.text}")


tariq_creds = create_principal("tariq")   # Product B — Data Engineer
ahmed_creds = create_principal("ahmed")   # Product A — Data Scientist (full gold)
layla_creds  = create_principal("layla")  # Product A — Data Scientist (single table)

---
## Step 4 — Define Catalog Roles and Principal Roles

| Catalog Role | Privilege Scope | Assigned To |
|---|---|---|
| `orders_full_access` | Read + Write on `bronze`, `silver`, `gold` | `ecomm_data_engineer` |
| `gold_namespace_reader` | Read on entire `gold` namespace | `analytics_data_scientist` |
| `gold_sales_table_reader` | Read on `gold.daily_sales_summary` table only | `inventory_data_scientist` |

In [ ]:
def create_catalog_role(role_name):
    r = requests.post(
        f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}/catalog-roles",
        headers=root_headers,
        json={"catalogRole": {"name": role_name}},
    )
    if r.status_code in (200, 201):
        print(f"Catalog role '{role_name}' created")
    elif r.status_code == 409:
        print(f"Catalog role '{role_name}' already exists")
    else:
        print(f"Failed to create catalog role '{role_name}': {r.text}")


def create_principal_role(role_name):
    r = requests.post(
        f"{POLARIS_URL}/api/management/v1/principal-roles",
        headers=root_headers,
        json={"principalRole": {"name": role_name}},
    )
    if r.status_code in (200, 201):
        print(f"Principal role '{role_name}' created")
    elif r.status_code == 409:
        print(f"Principal role '{role_name}' already exists")
    else:
        print(f"Failed to create principal role '{role_name}': {r.text}")


# Catalog roles
create_catalog_role("orders_full_access")       # Tariq: read + write on all layers
create_catalog_role("gold_namespace_reader")    # Ahmed: read entire gold namespace
create_catalog_role("gold_sales_table_reader")  # Layla: read one table only

print()

# Principal roles
create_principal_role("ecomm_data_engineer")       # Tariq
create_principal_role("analytics_data_scientist")  # Ahmed
create_principal_role("inventory_data_scientist")  # Layla

---
## Step 5 — Grant Privileges

Data engineers receive full read **and write** privileges on every namespace.
Data scientists receive scoped read-only grants — at either the namespace or table level.

In [ ]:
# Read-only: discover the namespace, list tables, and fetch row data
READ_PRIVS = [
    "NAMESPACE_READ_PROPERTIES",
    "TABLE_LIST",
    "TABLE_READ_DATA",
]

# Full engineering access: everything in READ_PRIVS plus mutation capabilities
WRITE_PRIVS = READ_PRIVS + [
    "TABLE_WRITE_DATA",
    "TABLE_CREATE",
    "TABLE_DROP",
]


def grant_namespace_privileges(namespace, catalog_role, privileges):
    """Grant a list of privileges on a namespace to a catalog role."""
    url = f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}/catalog-roles/{catalog_role}/grants"
    for priv in privileges:
        payload = {"grant": {"type": "namespace", "namespace": [namespace], "privilege": priv}}
        r = requests.put(url, headers=root_headers, json=payload)
        status = "OK" if r.status_code in (200, 201, 204) else f"FAILED ({r.status_code}: {r.text})"
        print(f"  {priv} on namespace '{namespace}' -> '{catalog_role}': {status}")


def grant_table_privileges(namespace, table, catalog_role, privileges):
    """Grant a list of privileges on a single table to a catalog role.

    The Polaris TableGrant schema requires the field name 'tableName', not 'name'.
    """
    url = f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}/catalog-roles/{catalog_role}/grants"
    for priv in privileges:
        payload = {
            "grant": {
                "type": "table",
                "namespace": [namespace],
                "tableName": table,
                "privilege": priv,
            }
        }
        r = requests.put(url, headers=root_headers, json=payload)
        status = "OK" if r.status_code in (200, 201, 204) else f"FAILED ({r.status_code}: {r.text})"
        print(f"  {priv} on table '{namespace}.{table}' -> '{catalog_role}': {status}")


def bind_roles(catalog_role, principal_role, principal_name):
    """Link: Principal -> Principal Role -> Catalog Role."""
    r1 = requests.put(
        f"{POLARIS_URL}/api/management/v1/principal-roles/{principal_role}/catalog-roles/{CATALOG_NAME}",
        headers=root_headers,
        json={"name": catalog_role},
    )
    r2 = requests.put(
        f"{POLARIS_URL}/api/management/v1/principals/{principal_name}/principal-roles",
        headers=root_headers,
        json={"name": principal_role},
    )
    if r1.ok and r2.ok:
        print(f"Bound: {principal_name} -> {principal_role} -> {catalog_role}")
    else:
        print(f"Failed to bind roles for '{principal_name}'")

In [ ]:
# --- Tariq (Product B Data Engineer) ---
# Read + write on all three medallion layers
print("Granting privileges for Tariq (Data Engineer):")
grant_namespace_privileges("bronze", "orders_full_access", WRITE_PRIVS)
grant_namespace_privileges("silver", "orders_full_access", WRITE_PRIVS)
grant_namespace_privileges("gold",   "orders_full_access", WRITE_PRIVS)
bind_roles("orders_full_access", "ecomm_data_engineer", "tariq")

print()

# --- Ahmed (Product A Data Scientist) ---
# Read-only on the entire gold namespace (all current and future tables)
print("Granting privileges for Ahmed (Data Scientist — full gold):")
grant_namespace_privileges("gold", "gold_namespace_reader", READ_PRIVS)
bind_roles("gold_namespace_reader", "analytics_data_scientist", "ahmed")

print()

# --- Layla (Product A Data Scientist) ---
# Read-only on gold.daily_sales_summary table ONLY.
# NAMESPACE_READ_PROPERTIES is granted at namespace level so the REST client
# can resolve the namespace path; TABLE_READ_DATA is table-scoped.
print("Granting privileges for Layla (Data Scientist — single table):")
grant_namespace_privileges("gold", "gold_sales_table_reader", ["NAMESPACE_READ_PROPERTIES"])
grant_table_privileges("gold", "daily_sales_summary", "gold_sales_table_reader", ["TABLE_READ_DATA"])
bind_roles("gold_sales_table_reader", "inventory_data_scientist", "layla")

---
## Step 6 — Build Per-Principal PyIceberg Catalogs

Each principal gets their own `RestCatalog` backed by their personal OAuth2 token.
Polaris evaluates every request against that principal's actual RBAC policy — no proxy in between.

With credential vending enabled, the flow on `load_table` is:
1. PyIceberg sends `X-Iceberg-Access-Delegation: vended-credentials`
2. Polaris checks RBAC — if denied, returns 403 before reaching S3
3. If allowed, Polaris calls the SeaweedFS STS endpoint to generate temporary credentials
4. PyIceberg receives those credentials and uses them for Parquet reads

In [ ]:
from pyiceberg.catalog.rest import RestCatalog


def make_catalog(principal_name, creds):
    """Build a PyIceberg RestCatalog authenticated as the given Polaris principal."""
    token = get_polaris_token(creds["clientId"], creds["clientSecret"])
    catalog = RestCatalog(
        name=f"{principal_name}_catalog",
        **{
            "uri": f"{POLARIS_URL}/api/catalog",
            "warehouse": CATALOG_NAME,
            "token": token,
            "s3.endpoint": S3_ENDPOINT,
            "s3.access-key-id": S3_ACCESS_KEY,
            "s3.secret-access-key": S3_SECRET_KEY,
            "s3.region": S3_REGION,
            "s3.path-style-access": "true",
        },
    )
    print(f"Catalog ready for '{principal_name}'")
    return catalog


tariq_catalog = make_catalog("tariq", tariq_creds)
ahmed_catalog = make_catalog("ahmed", ahmed_creds)
layla_catalog  = make_catalog("layla",  layla_creds)

---
## Step 7 — Verify: Namespace Listing

Expected results:

| Principal | `bronze` | `silver` | `gold` |
|---|---|---|---|
| `tariq` | ALLOWED | ALLOWED | ALLOWED |
| `ahmed` | DENIED | DENIED | ALLOWED |
| `layla` | DENIED | DENIED | ALLOWED (namespace visible, tables filtered) |

In [ ]:
def check_list_tables(principal_name, catalog, namespace):
    try:
        tables = catalog.list_tables(namespace)
        names = [t[-1] for t in tables]
        print(f"  [{principal_name}] {namespace}: ALLOWED — tables: {names}")
    except Exception as exc:
        print(f"  [{principal_name}] {namespace}: DENIED  — {exc}")


print("Tariq (Data Engineer) — expects all three layers:")
check_list_tables("tariq", tariq_catalog, "bronze")
check_list_tables("tariq", tariq_catalog, "silver")
check_list_tables("tariq", tariq_catalog, "gold")

print()

print("Ahmed (Data Scientist) — expects gold only:")
check_list_tables("ahmed", ahmed_catalog, "bronze")
check_list_tables("ahmed", ahmed_catalog, "silver")
check_list_tables("ahmed", ahmed_catalog, "gold")

print()

print("Layla (Data Scientist) — expects namespace-level denial on bronze/silver, limited gold:")
check_list_tables("layla", layla_catalog, "bronze")
check_list_tables("layla", layla_catalog, "silver")
check_list_tables("layla", layla_catalog, "gold")

---
## Step 8 — Verify: Table Load & Data Scan

Loading a table hits Polaris (RBAC enforced). If the principal is authorized, Polaris
returns vended temporary S3 credentials so PyIceberg can read the Parquet files.

In [ ]:
def scan_table(principal_name, catalog, namespace, table_name, limit=5):
    label = f"{namespace}.{table_name}"
    try:
        table = catalog.load_table((namespace, table_name))
        df = table.scan(limit=limit).to_pandas()
        print(f"  [{principal_name}] {label}: ALLOWED — {len(df)} row(s)")
        print(df.to_string(index=False))
    except Exception as exc:
        print(f"  [{principal_name}] {label}: DENIED  — {exc}")
    print()

In [ ]:
print("=== Tariq (Data Engineer) — full read + write access ===")
print()
scan_table("tariq", tariq_catalog, "bronze", "raw_orders")
scan_table("tariq", tariq_catalog, "silver", "cleansed_orders")
scan_table("tariq", tariq_catalog, "gold",   "daily_sales_summary")

In [ ]:
print("=== Ahmed (Data Scientist) — gold namespace only ===")
print()
scan_table("ahmed", ahmed_catalog, "gold",   "daily_sales_summary")  # ALLOWED
scan_table("ahmed", ahmed_catalog, "silver", "cleansed_orders")       # DENIED
scan_table("ahmed", ahmed_catalog, "bronze", "raw_orders")            # DENIED

In [ ]:
print("=== Layla (Data Scientist) — gold.daily_sales_summary table only ===")
print()
scan_table("layla", layla_catalog, "gold",   "daily_sales_summary")  # ALLOWED (single table grant)
scan_table("layla", layla_catalog, "silver", "cleansed_orders")       # DENIED
scan_table("layla", layla_catalog, "bronze", "raw_orders")            # DENIED

---
## Summary

| Table | Tariq (Engineer) | Ahmed (Scientist) | Layla (Scientist) |
|---|---|---|---|
| `bronze.raw_orders` | Read + Write | Denied | Denied |
| `silver.cleansed_orders` | Read + Write | Denied | Denied |
| `gold.daily_sales_summary` | Read + Write | Allowed | Allowed |
| Any future `gold.*` table | Read + Write | Allowed (namespace grant) | Denied (table grant only) |

Key design points:

1. **Namespace-level grants** (Ahmed, Tariq) automatically cover tables added in the future.
2. **Table-level grants** (Layla) are static — they must be updated explicitly when schema evolves.
3. `NAMESPACE_READ_PROPERTIES` is required even for table-level grantees so the REST client
   can resolve the namespace path.
4. Write privileges (`TABLE_WRITE_DATA`, `TABLE_CREATE`, `TABLE_DROP`) are additive on top
   of read privileges and do not affect data scientists' read-only roles.
5. **Credential vending** is enabled: Polaris calls the SeaweedFS STS endpoint on each
   `loadTable` to issue short-lived, path-scoped S3 tokens. RBAC denial happens before STS
   is called — no S3 credentials are issued to unauthorized principals.

---
## Cleanup (Optional)

In [ ]:
# Uncomment to clean up:

# for principal in ["tariq", "ahmed", "layla"]:
#     requests.delete(f"{POLARIS_URL}/api/management/v1/principals/{principal}", headers=root_headers)

# for p_role in ["ecomm_data_engineer", "analytics_data_scientist", "inventory_data_scientist"]:
#     requests.delete(f"{POLARIS_URL}/api/management/v1/principal-roles/{p_role}", headers=root_headers)

# for c_role in ["orders_full_access", "gold_namespace_reader", "gold_sales_table_reader"]:
#     requests.delete(
#         f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}/catalog-roles/{c_role}",
#         headers=root_headers,
#     )

# print("Cleanup complete")